In [3]:
! pip install pypandoc
import pypandoc




[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [59]:
"""Text normalization for fuzzy comparison."""

from __future__ import annotations

import html
import re
import string
import unicodedata
from typing import List, Optional

_BLOCK_TAG_PATTERN = re.compile(
    r"</?(?:p|br|div|h[1-6]|li|tr|blockquote|ul|ol|table|section|article)(?:\s[^>]*)?/?>",
    flags=re.IGNORECASE,
)
_TAG_PATTERN = re.compile(r"<[^>]+>")


def html_to_text(html_str: str) -> str:
    """Strip HTML tags and normalize whitespace."""
    if not html_str:
        return ""

    text = html.unescape(html_str)
    text = _BLOCK_TAG_PATTERN.sub(" ", text)
    text = _TAG_PATTERN.sub("", text)
    return re.sub(r"\s+", " ", text).strip()


SPECIAL_CHAR_TRANSLATION = str.maketrans(
    {
        "\u00a0": " ",
        "\u2009": " ",
        "\u200a": " ",
        "\u202f": " ",
        "\u2010": "-",
        "\u2011": "-",
        "\u2012": "-",
        "\u2013": "-",
        "\u2014": "-",
        "\u2212": "-",
        "\u2018": "'",
        "\u2019": "'",
        "\u201c": '"',
        "\u201d": '"',
        "\u00b5": "\u03bc",
    }
)


def normalize(
    s: str,
    do_not_remove: str = "",
    do: Optional[List[str]] = None,
) -> str:
    """Apply configurable normalization operations in a stable order."""
    if not s:
        return ""

    if do is None:
        do = [
            "ctrl",
            "strip",
            "lower",
            "html_unescape",
            "html_tags",
            "punctuation",
            "unicode",
            "special_chars",
        ]

    def remove_control_characters(
        txt: str,
        excluded: List[str] | None = None,
    ) -> str:
        excluded = excluded or ["Cc", "Cf"]
        cleaned = []
        for ch in txt:
            if ch in ("\n", "\r"):
                cleaned.append(ch)
            elif unicodedata.category(ch) not in excluded:
                cleaned.append(ch)
        return "".join(cleaned)

    if "ctrl" in do:
        s = remove_control_characters(s)

    if "strip" in do:
        s = s.strip()

    if "lower" in do:
        s = s.lower()

    if "html_unescape" in do:
        s = html.unescape(s)

    if "html_tags" in do:
        s = html_to_text(s)

    if "line_breaks" in do or "special_chars" in do:
        s = re.sub(r"(\S)(\n|\r)(\S)", r"\1 \3", s)
        s = s.replace("\n", " ").replace("\r", " ")

    if "special_chars" in do:
        s = s.translate(SPECIAL_CHAR_TRANSLATION)

    if "punctuation" in do:
        punctuation = string.punctuation
        for c in do_not_remove:
            punctuation = punctuation.replace(c, "")
        s = re.sub(f"[{re.escape(punctuation)}]", "", s)

    if "unicode" in do:
        s = (
            unicodedata.normalize("NFKD", s)
            .encode("ascii", "ignore")
            .decode("utf-8", "ignore")
        )

    return re.sub(r"\s+", " ", s).strip()


def normalize_text(
    text: str,
    strip_html: bool = True,
    keep_chars: str = "",
    config: Optional[dict] = None,
) -> str:
    """
    Normalize text for benchmark comparisons.

    Defaults mirror the caption verification path: strip/lowercase text,
    unescape and remove HTML, normalize line breaks and common typographic
    characters, and remove control characters. Punctuation and full ASCII
    Unicode folding are opt-in because punctuation can carry scientific
    meaning in captions.
    """
    if not text:
        return ""

    config = config or {}
    operations = ["strip", "lower", "line_breaks", "special_chars"]

    if strip_html:
        operations.extend(["html_tags", "html_unescape"])

    if config.get("normalize_unicode", False):
        operations.append("unicode")

    if config.get("remove_punctuation", False):
        operations.append("punctuation")

    if config.get("remove_control_chars", True):
        operations.append("ctrl")

    return normalize(text, do_not_remove=keep_chars, do=operations)

 

In [2]:
culprit = """<em>N</em><sup>1</sup>-methyladenosine
(m<sup>1</sup>A)"""

In [7]:
normalize(culprit, do=["html_tags", "html_unescape"])

'N1-methyladenosine(m1A)'

In [8]:
normalize_text(culprit)

'n1-methyladenosine(m1a)'

In [62]:
from pathlib import Path
path = Path("~/Desktop/EMBOJ-2025-121381R1-Manuscript_Text-mstxt.docx").expanduser()
result = pypandoc.convert_file(str(path), "html")
index = result.find('Figure legends')
subset = result[index:index+500]
print("\n\nORIGINAL:\n")
print(subset)
print("\n\nNORMALIZED:\n")
print(normalize_text(subset))




ORIGINAL:

Figure legends</strong></p>
<p><strong>Figure 1. CKD progression phenotypes observed upon increased
kidney load in systemic <em>Cdkal1</em> KO mice.</strong>
(<strong>A</strong>) Chemical structure of ms<sup>2</sup>t<sup>6</sup>A.
The modified residues of
2-methylthio-<em>N</em><sup>6</sup>-threonylcarbamoyladenosine
(ms<sup>2</sup>t<sup>6</sup>A) are depicted in red and the adenosine
backbone in black. (<strong>B</strong>) Secondary structure of the human
cytoplasmic tRNA<sup>Lys</sup><sub>UUU<


NORMALIZED:

figure legends figure 1. ckd progression phenotypes observed upon increased kidney load in systemic cdkal1 ko mice. (a) chemical structure of ms2t6a. the modified residues of 2-methylthio-n6-threonylcarbamoyladenosine (ms2t6a) are depicted in red and the adenosine backbone in black. (b) secondary structure of the human cytoplasmic trnalysuuu<


In [ ]:
result.find('Figure legends')

77401

In [51]:
subset =result[77401:77401+500]
print(subset)

Figure legends</strong></p>
<p><strong>Figure 1. CKD progression phenotypes observed upon increased
kidney load in systemic <em>Cdkal1</em> KO mice.</strong>
(<strong>A</strong>) Chemical structure of ms<sup>2</sup>t<sup>6</sup>A.
The modified residues of
2-methylthio-<em>N</em><sup>6</sup>-threonylcarbamoyladenosine
(ms<sup>2</sup>t<sup>6</sup>A) are depicted in red and the adenosine
backbone in black. (<strong>B</strong>) Secondary structure of the human
cytoplasmic tRNA<sup>Lys</sup><sub>UUU<


In [60]:
normalize_text(subset)

'figure legends figure 1. ckd progression phenotypes observed upon increased kidney load in systemic cdkal1 ko mice. (a) chemical structure of ms2t6a. the modified residues of 2-methylthio-n6-threonylcarbamoyladenosine (ms2t6a) are depicted in red and the adenosine backbone in black. (b) secondary structure of the human cytoplasmic trnalysuuu<'

In [ ]:
#find this string '(m<sup>1</sup>A)` in result
from IPython.display import display, HTML
display(HTML(result))